# 청년 정책 RAG 평가 실습

| 단계 | 내용 |
|------|------|
| 1 | 환경 설정 & 변수 확인 |
| 2 | 평가 질문 10개 + 기준 답변 확인 |
| 3 | LangSmith Dataset 생성 & 업로드 |
| 4 | `evaluate()` 실행 |
| 5 | 실패 케이스 분석 |

## Step 1. 환경 설정

In [1]:
import os, sys, json
from pathlib import Path

EVAL_DIR = Path().resolve()
ROOT = EVAL_DIR.parent
for p in [str(ROOT), str(EVAL_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

required = {
    "OPENAI_API_KEY":       "OpenAI API",
    "LANGSMITH_API_KEY":    "LangSmith API",
    "LANGSMITH_TRACING": "트레이싱 활성화 (true)",
    "LANGSMITH_PROJECT":    "LangSmith 프로젝트 이름",
}
print("환경변수 확인")
print("-" * 45)
all_ok = True
for k, desc in required.items():
    val = os.getenv(k, "")
    status = "✅" if val else "❌ 누락"
    if not val:
        all_ok = False
    preview = val[:8] + "..." if len(val) > 8 else val
    print(f"{status}  {k:<30} ({desc}) = {preview}")
print()
if all_ok:
    print("모든 환경변수 정상 설정됨")
else:
    print("⚠️  .env 파일에 누락된 값을 채워주세요.")

환경변수 확인
---------------------------------------------
✅  OPENAI_API_KEY                 (OpenAI API) = sk-proj-...
✅  LANGSMITH_API_KEY              (LangSmith API) = lsv2_pt_...
✅  LANGSMITH_TRACING              (트레이싱 활성화 (true)) = true
✅  LANGSMITH_PROJECT              (LangSmith 프로젝트 이름) = seoul-yo...

모든 환경변수 정상 설정됨


## Step 2. 평가 질문 10개 + 기준 답변 확인

In [2]:
DATASET_PATH = EVAL_DIR / "dataset.jsonl"
rows = [json.loads(l) for l in DATASET_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

print(f"총 {len(rows)}개 평가 예제\n")
print(f"{'#':<3} {'질문 (question)':<45} {'유형':<12} {'정답 정책 수'}")
print("-" * 75)
for i, r in enumerate(rows, 1):
    types = "/".join(set(p["type"] for p in r["expected_policies"]))
    n_pol = len(r["expected_policies"])
    print(f"{i:<3} {r['question'][:44]:<45} {types:<12} {n_pol}개")

총 10개 평가 예제

#   질문 (question)                                 유형           정답 정책 수
---------------------------------------------------------------------------
1   서울 관악구 사회초년생 청년 전세자금 대출 정책                    금융           2개
2   서울 신혼부부 공공임대 주택 정책                            주거           2개
3   서울 청년 주거 금융 종합 추천                             금융/주거        2개
4   서울 청년 월세 지원 보증금 부족                            금융           2개
5   서울 역세권 청년주택 대학원생                              주거           1개
6   서울 이주 예정 신혼부부 전세자금 대출                         금융           2개
7   서울 무주택 청년 1인가구 청약 분양                          주거           1개
8   서울 한부모 가구 임대주택 우선공급                           주거           1개
9   프리랜서 청년 전세자금 대출 소득증빙                          금융           2개
10  서울 왕십리역 인근 청년 직장인 임대 정책                       주거           2개


In [3]:
# 특정 예제 상세 확인 (IDX: 0~9)
IDX = 0
r = rows[IDX]
print(f"Q{IDX+1}. {r['question']}")
print(f"\n원본 발화: {r['input']}")
print(f"\nuser_profile:")
for k, v in r["user_profile"].items():
    print(f"  {k}: {v}")
print(f"\nground_truth:\n  {r['ground_truth']}")
print(f"\nexpected_policies:")
for p in r["expected_policies"]:
    print(f"  [{p['type']}] {p['title']} | 확인: {p['must_check']}")

Q1. 서울 관악구 사회초년생 청년 전세자금 대출 정책

원본 발화: 관악구에서 자취하려는 사회초년생인데 전세대출 뭐 있어?

user_profile:
  age: 27
  income_won: 32000000
  region: 서울 관악구
  no_house: True
  housing_type: 전세
  target: 청년

ground_truth:
  청년 사회초년생을 위한 서울시 청년 전월세보증금 이자지원 사업과 중소기업청년 전세자금대출, 버팀목 전세자금대출이 핵심. 만 19~34세 무주택 단독세대주 대상.

expected_policies:
  [금융] 청년 전월세보증금 이자지원 사업 | 확인: ['연소득', '무주택', '서울거주']
  [금융] 중소기업취업청년 전월세보증금대출 | 확인: ['재직', '연소득']


## Step 3. LangSmith Dataset 생성 & 업로드

In [4]:
from langsmith import Client

LS = Client()
DATASET_NAME = os.getenv("LS_DATASET", "youth-policy-eval-v1")

try:
    ds = LS.create_dataset(
        dataset_name=DATASET_NAME,
        description="서울 청년 주거·금융 정책 RAG 평가셋"
    )
    print(f"✅ 새 데이터셋 생성: {DATASET_NAME}")
    is_new = True
except Exception:
    ds = LS.read_dataset(dataset_name=DATASET_NAME)
    print(f"📂 기존 데이터셋 사용: {DATASET_NAME}")
    is_new = False

print(f"   dataset_id: {ds.id}")

if is_new:
    inputs  = [{"question": r["question"], "user_profile": r["user_profile"], "input": r["input"]} for r in rows]
    outputs = [{"ground_truth": r["ground_truth"], "expected_policies": r["expected_policies"]} for r in rows]
    LS.create_examples(inputs=inputs, outputs=outputs, dataset_id=ds.id)
    print(f"✅ {len(rows)}개 예제 업로드 완료")
else:
    existing = list(LS.list_examples(dataset_id=ds.id))
    print(f"   기존 예제 수: {len(existing)}개")

print(f"\n🔗 https://smith.langchain.com → Datasets → {DATASET_NAME}")

✅ 새 데이터셋 생성: youth-policy-eval-v1
   dataset_id: 988738f0-58ca-40ae-972a-3d9049c2c250
✅ 10개 예제 업로드 완료

🔗 https://smith.langchain.com → Datasets → youth-policy-eval-v1


## Step 4. evaluate() 실행

### 4-1. 파이프라인 초기화

In [5]:
from pipeline import run_pipeline, run_finance_pipeline
from chain.rag_chain import build_chain

print("파이프라인 초기화 중... (벡터DB 이미 있으면 빠름)")
housing_ret = run_pipeline()
finance_ret = run_finance_pipeline()
top3_chain, report_chain = build_chain(housing_ret, finance_ret)
print("✅ 준비 완료")

파이프라인 초기화 중... (벡터DB 이미 있으면 빠름)
🔄 PDF에서 Document 생성 시작


FileNotFoundError: [Errno 2] No such file or directory: './data/raw'

### 4-2. target_fn 정의 & 단건 테스트

In [ ]:
def target_fn(inputs: dict) -> dict:
    """LangSmith evaluate()가 각 예제에 대해 호출하는 함수."""
    q = inputs["question"]
    h_docs = housing_ret.invoke(q)[:5]
    f_docs = finance_ret.invoke(q)[:5]
    contexts = [d.page_content for d in (h_docs + f_docs)]
    answer = report_chain.invoke({"query": q})
    return {"answer": answer, "contexts": contexts}

# 단건 동작 확인
test_out = target_fn({"question": rows[0]["question"], "user_profile": rows[0]["user_profile"], "input": rows[0]["input"]})
print(f"answer 길이: {len(test_out['answer'])} chars")
print(f"contexts 수: {len(test_out['contexts'])}개")
print(f"answer 미리보기:\n{test_out['answer'][:300]}...")

### 4-3. 전체 10개 evaluate() 실행

In [ ]:
import datetime as dt
from langsmith.evaluation import evaluate as ls_evaluate
from evaluators import CUSTOM_EVALUATORS

exp_name = f"rag-v{dt.datetime.now().strftime('%Y%m%d-%H%M')}"
print(f"실험 이름: {exp_name}")
print(f"평가 지표: {[fn.__name__ for fn in CUSTOM_EVALUATORS]}")
print("실행 중... (10개 × 4개 지표, 약 1~2분 소요)\n")

results = ls_evaluate(
    target_fn,
    data=DATASET_NAME,
    evaluators=CUSTOM_EVALUATORS,
    experiment_prefix=exp_name,
    max_concurrency=2,
    metadata={"chain": "top3+report", "model": "gpt-4o-mini"},
)
print(f"\n✅ 실험 완료")

## Step 5. 실패 케이스 분석

In [ ]:
import pandas as pd

result_rows = []
for r in results:
    row = {"question": r["example"].inputs["question"]}
    for ev in r["evaluation_results"]["results"]:
        row[ev.key] = round(ev.score, 3) if ev.score is not None else None
    result_rows.append(row)

df = pd.DataFrame(result_rows)
score_cols = [c for c in df.columns if c != "question"]

pd.set_option("display.max_colwidth", 45)
pd.set_option("display.float_format", "{:.3f}".format)
print("평가 결과 전체")
display(df)

print("\n=== 지표별 평균 ===")
for col, val in df[score_cols].mean().items():
    bar = "█" * int(val * 20) + "░" * (20 - int(val * 20))
    print(f"  {col:<25} {bar}  {val:.3f}")

In [ ]:
THRESHOLD = 0.5

print(f"=== 실패 케이스 (점수 < {THRESHOLD}) ===")
found = False
for _, row in df.iterrows():
    failed = [(col, row[col]) for col in score_cols if row[col] is not None and row[col] < THRESHOLD]
    if failed:
        found = True
        print(f"\n❌ Q: {row['question']}")
        for col, score in failed:
            print(f"   {col}: {score:.3f}  ← 개선 필요")

if not found:
    print("모든 케이스가 threshold 이상입니다 🎉")

In [ ]:
HINTS = {
    "retrieval_relevance": [
        "→ config.py RETRIEVER_K 증가 (현재 10)",
        "→ BM25_WEIGHT / DENSE_WEIGHT 조정 (현재 0.85 / 0.2)",
        "→ MMR_LAMBDA 낮추기 (다양성 확보, 현재 0.9)",
        "→ RERANKER_TOP_N 증가 (현재 7)",
    ],
    "groundedness": [
        "→ 프롬프트: '문서에 없는 내용은 절대 추가 금지' 강화",
        "→ candidates 본문 길이 완화 (page_content[:700] → [:1000])",
        "→ retrieval_relevance 먼저 개선 후 재평가",
    ],
    "policy_fit": [
        "→ extract_info() target/category 추출 프롬프트 점검",
        "→ decompose_query() 쿼리 품질 확인",
        "→ retriever에 metadata 필터 추가 (지역, 대상)",
        "→ top3 선정 프롬프트의 선정 기준 강화",
    ],
    "freshness": [
        "→ metadata application_period 파싱 정확도 확인",
        "→ 만료된 정책 문서 벡터DB에서 제거 또는 날짜 필터",
    ],
}

print("=== 지표별 개선 방향 ===")
improved_needed = False
for col in score_cols:
    mean_score = df[col].mean()
    if mean_score < THRESHOLD:
        improved_needed = True
        print(f"\n🔧 {col} (평균 {mean_score:.3f})")
        for hint in HINTS.get(col, []):
            print(f"   {hint}")

if not improved_needed:
    print("개선이 필요한 지표가 없습니다 🎉")

In [ ]:
# 결과 로컬 저장 (재실행 없이 나중에 확인)
out_path = EVAL_DIR / "_last_run.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for r in results:
        rec = {
            "question":     r["example"].inputs["question"],
            "answer":       r["run"].outputs.get("answer", ""),
            "contexts":     r["run"].outputs.get("contexts", []),
            "ground_truth": r["example"].outputs.get("ground_truth", ""),
            "scores":       {ev.key: ev.score for ev in r["evaluation_results"]["results"]},
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✅ 결과 저장: {out_path}")